In [1]:
5+5

10

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F  

torch.manual_seed(0)
torch.set_printoptions(precision=2 , sci_mode=False, linewidth=120)

print(torch.__version__)


2.14.1+cpu


In [4]:
def show(mat, title="", row_labels=None, col_labels=None, decimals=2):
    """Print a 2-D tensor with token names on the rows and columns."""
    row_labels = row_labels or TOKENS
    col_labels = col_labels or TOKENS
    if title:
        print(title)
    print("       " + "".join(f"{c:>8}" for c in col_labels))
    for name, row in zip(row_labels, mat.tolist()):
        body = "".join(f"{val:8.{decimals}f}" for val in row)
        print(f"{name:>6} {body}")
    print()

# toy example

In [3]:
TOKENS  = ["the", "cat", "sat", "on", "mat"]

B       = 1
T       = len(TOKENS)
d_model = 8
n_heads = 2
d_k     = d_model // n_heads

x = torch.randn(B, T, d_model)

print("tokens :", TOKENS)
print("x.shape:", tuple(x.shape), " (batch, time, width)")
print()
print("x[0] -- one row per token:")
print(x)

tokens : ['the', 'cat', 'sat', 'on', 'mat']
x.shape: (1, 5, 8)  (batch, time, width)

x[0] -- one row per token:
tensor([[[-1.13, -1.15, -0.25, -0.43,  0.85,  0.69, -0.32, -2.12],
         [ 0.32, -1.26,  0.35,  0.31,  0.12,  1.24,  1.12, -0.25],
         [-1.35, -1.70,  0.57,  0.79,  0.60, -1.56, -0.34,  1.85],
         [-0.22, -0.74,  0.56,  0.26, -0.17, -0.68,  0.94,  0.49],
         [ 1.20,  0.08, -1.20, -0.00, -0.52, -0.31, -1.58,  1.71]]])


In [6]:
show(x[0])

            the     cat     sat      on     mat
   the    -1.13   -1.15   -0.25   -0.43    0.85    0.69   -0.32   -2.12
   cat     0.32   -1.26    0.35    0.31    0.12    1.24    1.12   -0.25
   sat    -1.35   -1.70    0.57    0.79    0.60   -1.56   -0.34    1.85
    on    -0.22   -0.74    0.56    0.26   -0.17   -0.68    0.94    0.49
   mat     1.20    0.08   -1.20   -0.00   -0.52   -0.31   -1.58    1.71



In [7]:
col_labels = [ f"d{i}" for i in range(5) ]
show(x[0][:, :5], "x, first 5 columns", col_labels=col_labels)

x, first 5 columns
             d0      d1      d2      d3      d4
   the    -1.13   -1.15   -0.25   -0.43    0.85
   cat     0.32   -1.26    0.35    0.31    0.12
   sat    -1.35   -1.70    0.57    0.79    0.60
    on    -0.22   -0.74    0.56    0.26   -0.17
   mat     1.20    0.08   -1.20   -0.00   -0.52



# compare tokens with dot product

In [8]:
naive_scores = x[0] @ x[0].T

print("x[0].shape:", tuple(x[0].shape))
print("naive_scores.shape:", tuple(naive_scores.shape))

show(naive_scores, "naive_scores[i][j] = how much token i matches token j")

x[0].shape: (5, 8)
naive_scores.shape: (5, 5)
naive_scores[i][j] = how much token i matches token j
            the     cat     sat      on     mat
   the     8.62    2.00   -1.39   -1.10   -4.91
   cat     2.00    4.77   -0.54    1.21   -2.77
   sat    -1.39   -0.54   11.98    3.61    1.41
    on    -1.10    1.21    3.61    2.59   -1.35
   mat    -4.91   -2.77    1.41   -1.35    8.67



## with Q K V

In [9]:
W_q = nn.Linear(d_model, d_model, bias=False)
W_k = nn.Linear(d_model, d_model, bias=False)
W_v = nn.Linear(d_model, d_model, bias=False)
W_o = nn.Linear(d_model, d_model)

q = W_q(x)
k = W_k(x)
v = W_v(x)

mask = torch.tril(
    torch.ones(T, T, dtype=torch.bool)
)

print("x.shape:", tuple(x.shape))
print("q.shape:", tuple(q.shape))
print("k.shape:", tuple(k.shape))
print("v.shape:", tuple(v.shape))
print()

x.shape: (1, 5, 8)
q.shape: (1, 5, 8)
k.shape: (1, 5, 8)
v.shape: (1, 5, 8)



In [10]:
scores = q @ k.transpose(-2, -1)

print("scores.shape:", tuple(scores.shape))
print()
show(scores[0], "scores = q @ k^T")

scores.shape: (1, 5, 5)

scores = q @ k^T
            the     cat     sat      on     mat
   the     0.70   -1.33    0.45   -0.22    0.33
   cat    -0.69   -0.17   -1.03   -0.20    0.78
   sat    -0.51   -0.31   -0.34   -0.35    0.07
    on    -0.45    0.47   -0.55   -0.03   -0.27
   mat    -0.87   -0.58   -0.46   -0.53    1.74



## Scale by sqrt(d_k)

In [11]:
def row(t, style="{:6.2f}"):
    return "[" + "  ".join(style.format(v) for v in t.tolist()) + "]"

small = torch.tensor([0.5, 1.2, 0.8])
big = small * 16

print("scores", row(small), "-> softmax", row(F.softmax(small, dim=-1), "{:6.1%}"))
print("scores", row(big),   "-> softmax", row(F.softmax(big,   dim=-1), "{:6.1%}"))

scores [  0.50    1.20    0.80] -> softmax [ 22.9%   46.1%   30.9%]
scores [  8.00   19.20   12.80] -> softmax [  0.0%   99.8%    0.2%]


In [13]:
scaled = scores / d_k **0.5

scaled

tensor([[[ 0.35, -0.67,  0.22, -0.11,  0.16],
         [-0.34, -0.09, -0.51, -0.10,  0.39],
         [-0.26, -0.15, -0.17, -0.18,  0.04],
         [-0.23,  0.24, -0.27, -0.01, -0.14],
         [-0.44, -0.29, -0.23, -0.26,  0.87]]], grad_fn=<DivBackward0>)

In [14]:
print("std of raw scores   :", round(scores.std().item(), 3))
print("std of scaled scores:", round(scaled.std().item(), 3))
print()
show(scaled[0], "scaled = scores / sqrt(d_k)")

std of raw scores   : 0.648
std of scaled scores: 0.324

scaled = scores / sqrt(d_k)
            the     cat     sat      on     mat
   the     0.35   -0.67    0.22   -0.11    0.16
   cat    -0.34   -0.09   -0.51   -0.10    0.39
   sat    -0.26   -0.15   -0.17   -0.18    0.04
    on    -0.23    0.24   -0.27   -0.01   -0.14
   mat    -0.44   -0.29   -0.23   -0.26    0.87



## The causal mask

In [15]:
max_seq_len = 16

sent = torch.ones(max_seq_len, max_seq_len, dtype=torch.bool)
sent_mask = torch.tril(sent)

print("sent.shape:", tuple(sent.shape))
print("mask.shape:", tuple(sent_mask.shape))

show(sent[:T, :T].int())
print()
show(sent_mask[:T, :T].int())

sent.shape: (16, 16)
mask.shape: (16, 16)
            the     cat     sat      on     mat
   the     1.00    1.00    1.00    1.00    1.00
   cat     1.00    1.00    1.00    1.00    1.00
   sat     1.00    1.00    1.00    1.00    1.00
    on     1.00    1.00    1.00    1.00    1.00
   mat     1.00    1.00    1.00    1.00    1.00


            the     cat     sat      on     mat
   the     1.00    0.00    0.00    0.00    0.00
   cat     1.00    1.00    0.00    0.00    0.00
   sat     1.00    1.00    1.00    0.00    0.00
    on     1.00    1.00    1.00    1.00    0.00
   mat     1.00    1.00    1.00    1.00    1.00



In [16]:
masked = scaled.masked_fill(~mask, float("-inf"))

show(masked[0], "scaled scores after masked_fill")

scaled scores after masked_fill
            the     cat     sat      on     mat
   the     0.35    -inf    -inf    -inf    -inf
   cat    -0.34   -0.09    -inf    -inf    -inf
   sat    -0.26   -0.15   -0.17    -inf    -inf
    on    -0.23    0.24   -0.27   -0.01    -inf
   mat    -0.44   -0.29   -0.23   -0.26    0.87



# Softmax, then mix the values

In [17]:
weights = F.softmax(masked, dim=-1) # (B, T, T)
out = weights @ v # (B, T, T) @ (B, T, d_model) => (B, T, d_model)

print("weights.shape:", tuple(weights.shape))
print("v.shape      :", tuple(v.shape))
print("out.shape    :", tuple(out.shape), " <- same shape we started with")
print()
show(weights[0], "attention weights (each row sums to 1)")

weights.shape: (1, 5, 5)
v.shape      : (1, 5, 8)
out.shape    : (1, 5, 8)  <- same shape we started with

attention weights (each row sums to 1)
            the     cat     sat      on     mat
   the     1.00    0.00    0.00    0.00    0.00
   cat     0.44    0.56    0.00    0.00    0.00
   sat     0.31    0.35    0.34    0.00    0.00
    on     0.21    0.33    0.20    0.26    0.00
   mat     0.12    0.14    0.15    0.14    0.45



# Checkpoint: wrap one head in a function

In [18]:
def single_head_attention(x, W_q, W_k, W_v, mask):
    """One attention head.

    x -> (B, T, d_model);  returns (B, T, d_model) and the attention weights.
    """

    B, T, d_model = x.shape

    q = W_q(x)                                          # Part 2
    k = W_k(x)
    v = W_v(x)

    scores = q @ k.transpose(-2, -1)                    # Part 2
    scores = scores / d_model**0.5                      # Part 3

    causal = mask[:T, :T]                               # Part 4
    scores = scores.masked_fill(~causal, float("-inf"))

    weights = F.softmax(scores, dim=-1)
    out = weights @ v

    return out, weights

In [19]:
head_out, head_w = single_head_attention(x, W_q, W_k, W_v, mask)

print("in :", tuple(x.shape))
print("out:", tuple(head_out.shape))

in : (1, 5, 8)
out: (1, 5, 8)


In [21]:
head_w

tensor([[[1.00, 0.00, 0.00, 0.00, 0.00],
         [0.45, 0.55, 0.00, 0.00, 0.00],
         [0.32, 0.34, 0.34, 0.00, 0.00],
         [0.22, 0.31, 0.21, 0.26, 0.00],
         [0.14, 0.16, 0.17, 0.16, 0.36]]], grad_fn=<SoftmaxBackward0>)

In [20]:
head_out

tensor([[[-1.00,  0.40, -0.70, -0.28,  0.74, -0.51, -0.58,  0.05],
         [-0.29, -0.01, -0.53, -0.21,  0.73, -0.55, -0.74, -0.19],
         [-0.44, -0.07, -0.58, -0.16,  0.50, -0.21, -0.46, -0.34],
         [-0.24, -0.17, -0.54, -0.12,  0.44, -0.20, -0.41, -0.25],
         [-0.05,  0.04,  0.05,  0.01, -0.09,  0.09, -0.15, -0.38]]], grad_fn=<UnsafeViewBackward0>)

# One head is not enough

In [22]:
head_a, head_b = None, None

torch.manual_seed(101)
head_a = [
    nn.Linear(d_model, d_model, bias=False)
    for _ in range(3)
]

torch.manual_seed(202)
head_b = [
    nn.Linear(d_model, d_model, bias=False)
    for _ in range(3)
]

o_a, w_a = single_head_attention(x, *head_a, mask)
o_b, w_b = single_head_attention(x, *head_b, mask)

show(w_a[0], "head A")
show(w_b[0], "head B")

head A
            the     cat     sat      on     mat
   the     1.00    0.00    0.00    0.00    0.00
   cat     0.48    0.52    0.00    0.00    0.00
   sat     0.44    0.38    0.18    0.00    0.00
    on     0.28    0.25    0.23    0.24    0.00
   mat     0.35    0.25    0.07    0.15    0.17

head B
            the     cat     sat      on     mat
   the     1.00    0.00    0.00    0.00    0.00
   cat     0.46    0.54    0.00    0.00    0.00
   sat     0.55    0.31    0.14    0.00    0.00
    on     0.26    0.27    0.22    0.25    0.00
   mat     0.25    0.12    0.08    0.12    0.42



In [23]:
def multi_head_attention(x, W_q, W_k, W_v, W_o, mask, n_heads):
    """Multi-head causal self-attention.  (B, T, d_model) -> (B, T, d_model)"""

    B, T, d_model = x.shape
    d_k = d_model // n_heads

    q = W_q(x).view(B, T, n_heads, d_k).transpose(1, 2)
    k = W_k(x).view(B, T, n_heads, d_k).transpose(1, 2) # => (B, n_heads, T, d_k)
    v = W_v(x).view(B, T, n_heads, d_k).transpose(1, 2)

    scores = q @ k.transpose(-2, -1)
    scores = scores / d_k**0.5
    scores = scores.masked_fill(~mask[:T, :T], float("-inf"))

    weights = F.softmax(scores, dim=-1)
    out = weights @ v

    out = out.transpose(1, 2).contiguous().view(B, T, d_model)
    # .transpose(1, 2) ===> (B, T, n_heads, d_k)
    # .view(B, T, d_model) ===> (B, T, d_model)

    return W_o(out), weights

In [24]:
mha_out, mha_w =  multi_head_attention(x, W_q, W_k, W_v, W_o, mask, n_heads)


print("in :", tuple(x.shape))
print("out:", tuple(mha_out.shape))
print("weights:", tuple(mha_w.shape), " (B, n_heads, T, T)")

in : (1, 5, 8)
out: (1, 5, 8)
weights: (1, 2, 5, 5)  (B, n_heads, T, T)


In [ ]:
class MyMultiHeadAttention(nn.Module):
    """Every token looks at every earlier token, through several lenses at once.

    Input:  (B, T, d_model)
    Output: (B, T, d_model)
    """

    def __init__(self, config):
        super().__init__()

        self.n_heads = config.n_heads
        self.d_k = config.d_k

        # One full-width projection per role, carved into heads in forward().
        self.W_q = nn.Linear(config.d_model, config.d_model, bias=False)
        self.W_k = nn.Linear(config.d_model, config.d_model, bias=False)
        self.W_v = nn.Linear(config.d_model, config.d_model, bias=False)

        self.W_o = nn.Linear(config.d_model, config.d_model)

        mask = torch.tril(
            torch.ones(config.max_seq_len, config.max_seq_len, dtype=torch.bool)
        )

        self.register_buffer("mask", mask, persistent=False)

    def forward(self, x):

        batch_size, seq_len, d_model = x.shape

        q = self.W_q(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        k = self.W_k(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)

        scores = q @ k.transpose(-2, -1)
        scores = scores / self.d_k**0.5

        scores = scores.masked_fill(~mask, float("-inf"))

        weights = F.softmax(scores, dim=-1)
        out = weights @ v

        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)

        return self.W_o(out), weights
        